# Adobe 10-K Document Loader

Load HTML files from `data/html/`, clean with BeautifulSoup, convert to markdown.


In [1]:
from pathlib import Path

HTML_DIR = Path("data/html")
DATA_DIR = Path("data")

html_files = sorted(HTML_DIR.glob("*.html"))
print(f"Found {len(html_files)} HTML files: {[f.name for f in html_files]}")


Found 4 HTML files: ['adobe_10k_fy2022.html', 'adobe_10k_fy2023.html', 'adobe_10k_fy2024.html', 'adobe_10k_fy2025.html']


In [2]:
from utils import clean_and_convert_html_2_markdown

for html_file in html_files:
    md_filename = html_file.stem + ".md"
    md_filepath = DATA_DIR / md_filename

    raw_html = html_file.read_text(encoding="utf-8")
    markdown = clean_and_convert_html_2_markdown(raw_html)

    md_filepath.write_text(markdown, encoding="utf-8")
    print(f"{html_file.name} -> {md_filename} ({len(markdown):,} characters)")


# Preview a sample from one of the converted files
sample = Path("data/adobe_10k_fy2024.md").read_text(encoding="utf-8")
print(f"\n\nSample:\n{sample[20_000:21_000]}")


adobe_10k_fy2022.html -> adobe_10k_fy2022.md (397,559 characters)
adobe_10k_fy2023.html -> adobe_10k_fy2023.md (392,649 characters)
adobe_10k_fy2024.html -> adobe_10k_fy2024.md (403,098 characters)
adobe_10k_fy2025.html -> adobe_10k_fy2025.md (371,252 characters)


Sample:
ve, interactive tutorials with creators. Additionally, with Adobe GenStudio and Adobe GenStudio for Performance Marketing, a generative AI-first product that natively integrates Digital Media and Digital Experience offerings, we help enterprises to quickly create on-brand content variations and accelerate their marketing workflows. Further descriptions of our Digital Media products are included below under “Principal Products, Services and Solutions.”

In our Creative Cloud business, we employ our product-led growth strategy to minimize the friction of customer interactions and drive positive product experiences, which results in increasing adoption, usage, conversion, expansion and loyalty. We also continue to emplo

In [3]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from chunker import ContextualChunker

load_dotenv()

# Load markdown files as LangChain Documents
md_files = sorted(Path("data").glob("adobe_10k_*.md"))
documents = []
for md_file in md_files:
    text = md_file.read_text(encoding="utf-8")
    documents.append(Document(page_content=text, metadata={"source": md_file.name}))

print(f"Loaded {len(documents)} documents")

# Chunk with context
chunker = ContextualChunker(chunk_size=1024, chunk_overlap=128)
chunks = chunker.chunk(documents)

print(f"\nTotal chunks: {len(chunks)}")
print(f"\nSample chunk (index 50):\n{'=' * 60}")
print(chunks[50].page_content[:600])


Loaded 4 documents
  [418/418] processed (cache hits: 418)
Done: 418 chunks (418 from cache, 0 generated).

Total chunks: 418

Sample chunk (index 50):
[Chunk_Context_Start]The preceding text discussed the company's stock repurchase program, including the authority granted by the Board of Directors to repurchase shares and the rationale behind the program, as well as recent legislative tax changes that could impact the company. This chunk provides a detailed account of the company's (likely Adobe's, based on context) stock repurchase activities during fiscal 2022, including the number of shares repurchased, the financial arrangements involved, and references to further details in other sections of the report. It also addresses the company's i


In [4]:
from vectorstore import WeaviateStore

store = WeaviateStore(collection_name="Adobe10K")
store.ingest(chunks)

# Test a search
results = store.search("What was Adobe's total revenue in fiscal year 2024?")
for i, r in enumerate(results):
    print(f"\n--- Result {i + 1} (source: {r['source']}) ---")
    print(r["content"][:400])

store.close()


Collection already has 418 objects — skipping ingestion.

--- Result 1 (source: adobe_10k_fy2024.md) ---
[Chunk_Context_Start]The preceding text discussed Adobe's strong revenue growth across its Creative and Document Cloud segments, highlighting year-over-year increases and introducing the Digital Experience segment as a key area of market leadership and innovation, particularly through AI integration. This chunk provides a detailed overview of Adobe's Digital Experience Cloud for fiscal year 2024, 

--- Result 2 (source: adobe_10k_fy2024.md) ---
[Chunk_Context_Start]The preceding text provided a summary of Adobe's financial performance for fiscal year 2024, highlighting key metrics such as annual recurring revenue, segment revenues, cost of revenue, and operating expenses. This chunk continues the financial overview by detailing Adobe's net income, cash flows from operations, and remaining performance obligations for fiscal 2024, followed

--- Result 3 (source: adobe_10k_fy2024.md) -